# Single Signal Analysis — PDF Analysis

This notebook analyzes the probability density functions (PDFs) of individual seismic signals. For each signal, the analysis includes: empirical PDF estimation, Gaussian fit with normality testing (Anderson-Darling), and heavy-tail characterization (Lévy, Student-t, power-law).

**Dataset:** Preprocessed signals (baseline-corrected, normalized) from notebook `01b_signals`.

**Outputs:**
- Individual PDF plots per signal
- Gaussian fit results with kurtosis, skewness, Anderson-Darling test
- Heavy-tail assessment (AIC comparison, power-law exponents)
- LaTeX tables for thesis integration

## 1. Imports and visualization settings

In [1]:
from pathlib import Path
import pandas as pd
import logging
from src import (
    plot_empirical_pdfs,
    gaussian_fit_analysis,
    heavy_tail_assessment,
    heavy_tail_to_latex,
    set_plot_style
    )
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

INFO | Environment ready


## 2. Configuration

Data type selection (acceleration, velocity, or displacement) and derivation of type-specific parameters: signal column name and physical unit. Output directories for figures, processed data, and LaTeX tables are also defined.

In [ ]:
# CONFIGURATION
# EVENT_ID = 'INT-41004391'
EVENT_ID = 'IT-2009-0009'
DATA_TYPE = 'displacement'  # Options: 'acceleration', 'velocity', 'displacement'

# Determine signal column name based on DATA_TYPE
if DATA_TYPE == 'acceleration':
    SIGNAL_COLUMN = 'acceleration'
    SIGNAL_UNIT = 'cm/s²'
elif DATA_TYPE == 'velocity':
    SIGNAL_COLUMN = 'velocity'
    SIGNAL_UNIT = 'cm/s'
elif DATA_TYPE == 'displacement':
    SIGNAL_COLUMN = 'displacement'
    SIGNAL_UNIT = 'cm'
else:
    raise ValueError(f"Unknown DATA_TYPE: {DATA_TYPE}")

logger.info(f"Working with {DATA_TYPE} data")

In [ ]:
# Get project root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Define all paths from project root
DATA_INPUT= PROJECT_ROOT / 'data' / 'processed' / EVENT_ID / '01b_signals' / DATA_TYPE
DATA_OUTPUT= PROJECT_ROOT / 'data' / 'processed' / EVENT_ID / '02_signals' / DATA_TYPE
FIGURES_DIR = PROJECT_ROOT / 'figures' / EVENT_ID / '02_pdf_analysis' / DATA_TYPE
LATEX_TABLES_DIR = PROJECT_ROOT / 'data' / 'processed' / EVENT_ID / 'latex_tables' / DATA_TYPE

# Create output directories
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
LATEX_TABLES_DIR.mkdir(parents=True, exist_ok=True)

check(FIGURES_DIR.exists(), f"Figures directory ready: {FIGURES_DIR}")
check(LATEX_TABLES_DIR.exists(), f"LaTeX tables directory ready: {LATEX_TABLES_DIR}")

## 3. Data Loading

Preprocessed signals are loaded from parquet files generated in notebook `01b_signals`. Each signal has been:
- **Baseline-corrected:** Mean removed ($\bar{a} = 0$)
- **Normalized:** Standardized by standard deviation ($\sigma_a = 1$)

This normalization allows direct comparison of distribution shapes across stations with different amplitudes.

In [ ]:
logger.info("Loading preprocessed data...")
df_signals_clean = pd.read_parquet(DATA_INPUT/ f'{DATA_TYPE[:3]}_preprocessed_pdf.parquet')

check(len(df_signals_clean) > 0, "Data loaded successfully")
logger.info(f"Shape: {df_signals_clean.shape} — {df_signals_clean['file'].nunique()} signals")

display(df_signals_clean.head())

Shape: (2614815, 4)


,file,sample,acceleration,acceleration_normalized
0,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,0,-6.666667e-10,-3.401661e-08
1,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,1,-6.666667e-10,-3.401661e-08
2,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,2,-6.666667e-10,-3.401661e-08
3,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,3,-6.666667e-10,-3.401661e-08
4,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,4,-6.666667e-10,-3.401661e-08


## 4. PDF analysis

Probability density functions are estimated using histogram binning and analyzed in three stages:
1. **Empirical PDF** — histogram estimation for each signal
2. **Gaussian fit** — test for normality using Anderson-Darling test
3. **Heavy-tail assessment** — compare Gaussian vs. heavy-tailed distributions

### 4.1 Empirical PDF per signal

One PDF plot is produced for each signal and saved in the 
figures directory.

In [ ]:
logger.info("Plotting empirical PDFs...")
plot_empirical_pdfs(
    df_signals_clean,
    signal_column=SIGNAL_COLUMN,
    signal_unit=SIGNAL_UNIT,
    bins=100,
    log_scale=True,
    normalized=True,
    output_dir=FIGURES_DIR / 'pdf_single',
    prefix=DATA_TYPE[:3]
)

Saved: 66/66 PDFs
All PDFs saved successfully!


### 4.2 Gaussian Fit and Normality Assessment

A Gaussian distribution $\mathcal{N}(\mu, \sigma^2)$ is fitted to each signal using maximum likelihood estimation. Normality is tested using the **Anderson-Darling (AD) test**, which is more sensitive to tail deviations than the Kolmogorov-Smirnov test.

#### Anderson-Darling Test

The test statistic is defined as:

$$A^2 = -n - \frac{1}{n} \sum_{i=1}^{n} (2i-1) \left[ \ln F(x_i) + \ln(1 - F(x_{n+1-i})) \right]$$

where:
- $F$ is the CDF of the fitted Gaussian
- $x_1 \leq \ldots \leq x_n$ are the ordered observations
- **Null hypothesis:** Data follows a Gaussian distribution
- **Rejection criterion:** $A^2 > A^2_{\text{crit}}$ at $\alpha = 0.05$

#### Distribution Shape Metrics

**Kurtosis** (excess, Fisher's definition):
$$\kappa = \frac{\mu_4}{\sigma^4} - 3$$

- $\kappa = 0$: Gaussian (mesokurtic)
- $\kappa > 0$: Heavy tails (leptokurtic)
- $\kappa < 0$: Light tails (platykurtic)

**Skewness:**
$$\gamma = \frac{\mu_3}{\sigma^3}$$

- $\gamma = 0$: Symmetric distribution
- $\gamma \neq 0$: Asymmetric (positive = right-skewed, negative = left-skewed)

where $\mu_k = \mathbb{E}[(a - \bar{a})^k]$ is the $k$-th central moment.

#### Outputs

For each signal, the function generates:
1. **PDF plot** with fitted Gaussian overlay
2. **Q-Q plot** to visualize deviations from normality
3. **Summary DataFrame** with $\mu$, $\sigma$, $\kappa$, $\gamma$, $A^2$ statistic

**Output directory:** `figures/02_pdf_analysis/{signal_type}/gaussian_fit/`

In [ ]:
logger.info("Running Gaussian fit analysis...")
df_gaussian_results = gaussian_fit_analysis(
    df_signals_clean,
    signal_column=SIGNAL_COLUMN,
    signal_unit=SIGNAL_UNIT,
    bins=100,
    log_scale=True,
    normalized=True,
    output_dir=FIGURES_DIR / 'gaussian_fit',
    prefix=DATA_TYPE[:3]
)

Saved: 66/66 individual plots
All individual plots saved successfully!
Non-Gaussian signals (AD test, p<5%): 66/66


In [ ]:
# Display Gaussian fit results
pd.set_option('display.max_rows', None)
display(df_gaussian_results[['station', 'stream', 'kurtosis', 'skewness', 'ad_statistic', 'ad_critical_5pct', 'non_gaussian']])
pd.reset_option('display.max_rows')

In [ ]:
# Save Gaussian fit results
logger.info("Saving Gaussian fit results...")
try:
    df_gaussian_results.to_parquet(DATA_OUTPUT / 'gaussian_fit_results.parquet', index=False)
    logger.info("Saved: gaussian_fit_results.parquet")
except Exception as e:
    logger.error(f"Error saving file: {e}")

### 4.3 Heavy-Tail Assessment

Seismic signals often exhibit **heavy-tailed distributions** due to strong transient events (P/S arrivals, aftershocks) and complex nonlinear dynamics. This section compares four distribution models using likelihood-based selection criteria.

#### Distribution Models

**1. Gaussian** (exponential tails):
$$p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

- Finite moments of all orders
- Tail decay: $p(x) \sim \exp(-x^2)$

**2. Laplace** (exponential tails, sharper peak):
$$p(x) = \frac{1}{2b} \exp\left(-\frac{|x-\mu|}{b}\right)$$

- Kurtosis $\kappa = 3$ (heavier than Gaussian)
- Tail decay: $p(x) \sim \exp(-|x|)$

**3. Student-t** (power-law tails):
$$p(x) = \frac{\Gamma(\frac{\nu+1}{2})}{\sqrt{\nu\pi}\,\Gamma(\frac{\nu}{2})} \left(1 + \frac{x^2}{\nu}\right)^{-\frac{\nu+1}{2}}$$

- Parameter: $\nu$ (degrees of freedom)
- Tail decay: $p(x) \sim x^{-(\nu+1)}$ for $|x| \to \infty$
- Lower $\nu$ → heavier tails
- $\nu \to \infty$ → Gaussian

**4. Lévy stable** (most general, includes Gaussian):
$$p(x) \sim x^{-(\alpha+1)} \quad \text{for } |x| \to \infty$$

- Parameters: $\alpha$ (stability), $\beta$ (skewness)
- $\alpha = 2$: Gaussian
- $\alpha < 2$: Power-law tails, infinite variance
- Fits via maximum likelihood estimation (MLE)

#### Model Selection

Models are compared using **Akaike Information Criterion (AIC)**:

$$\text{AIC} = 2k - 2\ln(\mathcal{L})$$

where:
- $k$ = number of parameters
- $\mathcal{L}$ = maximum likelihood

**Lower AIC** indicates better fit with penalty for model complexity.

#### Power-Law Tail Exponent

The **Hill estimator** is used to estimate the tail exponent from the upper 10% of $|x|$ values:

$$\hat{\alpha} = \left[ \frac{1}{n} \sum_{i=1}^{n} \ln\frac{x_i}{x_{\min}} \right]^{-1}$$

where $x_1 \geq x_2 \geq \ldots \geq x_n$ are the ordered tail observations above threshold $x_{\min}$.

#### Physical Interpretation

- **$\alpha < 2$:** Infinite variance → **anomalous diffusion** regime
- **$\alpha \approx 1.5$:** Common in seismic coda and turbulence
- **$\alpha \geq 3$:** Finite third moment exists

#### Outputs

For each signal:
1. **PDF comparison plot** with all four models overlaid
2. **Q-Q plot** vs. Gaussian
3. **Log-log CCDF** with power-law fit
4. **Summary DataFrame** with AIC values, best-fit model, tail exponents

**Output directory:** `figures/02_pdf_analysis/{signal_type}/heavy_tail/`

#### Computational Note

Lévy stable fitting is computationally expensive (MLE on 4 parameters). For signals longer than 5000 samples, a random subsample is used for fitting while the full signal is used for evaluation.

In [ ]:
# Check for partial results from heavy tail assessment
logger.info("Checking for partial results from heavy tail assessment")
try:
    partial_results_path = DATA_OUTPUT / f'heavy_tail_results_partial_{DATA_TYPE[:3]}.parquet'
    df_partial = pd.read_parquet(partial_results_path)
    logger.info(f"Partial results found: {len(df_partial)} signals already processed")
except Exception:
    logger.info("No partial results found, starting from scratch")

Partial results available: 66 signals already processed


In [ ]:
# Heavy-tail assessment
logger.info("Running heavy-tail assessment")
df_heavy_tail_results = heavy_tail_assessment(
    df_signals_clean,
    signal_column=SIGNAL_COLUMN,
    signal_unit=SIGNAL_UNIT,
    output_dir=FIGURES_DIR / 'heavy_tail',
    prefix=DATA_TYPE[:3],
    resume=True,
    partial_path = partial_results_path
)

Resuming from 66 already processed signals
Saved: 0/66 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit_aic
Levy-stable    55
Student-t      11
Name: count, dtype: int64


In [7]:
# Display heavy-tail assessment results
pd.set_option('display.max_rows', None)
display(df_heavy_tail_results[['station', 'stream', 'aic_gaussian', 'aic_laplace', 
                                'aic_student_t', 'aic_levy_stable', 'best_fit_aic', 'student_t_df', 
                                'power_law_exp']])
pd.reset_option('display.max_rows')

,station,stream,aic_gaussian,aic_laplace,aic_student_t,aic_levy_stable,best_fit_aic,student_t_df,power_law_exp
0,EILF,HNE,136221.10,75859.86,52804.13,52746.79,Levy-stable,1.2651,1.1420
1,EILF,HNN,136221.10,68603.98,33000.83,33018.55,Student-t,0.9964,1.1200
2,EILF,HNZ,136221.10,85179.23,67682.48,67581.97,Levy-stable,1.3937,1.1692
3,ESCA,HNE,136221.10,46302.74,-41994.24,-42357.24,Levy-stable,0.6018,1.0875
4,ESCA,HNN,136221.10,51314.78,-35903.50,-36218.17,Levy-stable,0.6116,1.0638
5,ESCA,HNZ,136221.10,52175.36,-28865.09,-28829.31,Student-t,0.6807,1.0188
6,ISO,HNE,136221.10,19666.20,-100500.55,-100096.19,Student-t,0.7025,0.7322
7,ISO,HNN,136221.10,23524.52,-88695.53,-88301.55,Student-t,0.7135,0.7902
8,ISO,HNZ,136221.10,14731.63,-112766.34,-112240.15,Student-t,0.7140,0.6845
9,MFC,HNE,136221.10,60178.50,-27107.33,-28590.05,Levy-stable,0.5136,1.1546


In [ ]:
# Save heavy-tail assessment results
logger.info("Saving heavy-tail results")
try:
    output_path = DATA_OUTPUT / f'heavy_tail_results_{DATA_TYPE[:3]}.parquet'
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df_heavy_tail_results.to_parquet(output_path, index=False)
    logger.info(f"Saved: {output_path}")
except Exception as e:
    logger.error(f"Error saving file: {e}")

Saved successfully!


In [ ]:
# Generate LaTeX table for heavy-tail assessment results
logger.info("Generating LaTeX table for heavy-tail assessment...")
latex_rows = heavy_tail_to_latex(
    df_heavy_tail_results,
    output_path=LATEX_TABLES_DIR / f'heavy_tail_table_{DATA_TYPE[:3]}.tex'
)
logger.info(f"Saved: heavy_tail_table_{DATA_TYPE[:3]}.tex")

Saved to: ../data/processed/heavy_tail_table.tex
EILF & HNE & 52,746.79 & 52,804.13 & L\'evy-stable & 1.2651 & 1.1420 \\
EILF & HNN & 33,018.55 & 33,000.83 & Student-$t$ & 0.9964 & 1.1200 \\
EILF & HNZ & 67,581.97 & 67,682.48 & L\'evy-stable & 1.3937 & 1.1692 \\
\addlinespace
ESCA & HNE & $-$42,357.24 & $-$41,994.24 & L\'evy-stable & 0.6018 & 1.0875 \\
ESCA & HNN & $-$36,218.17 & $-$35,903.50 & L\'evy-stable & 0.6116 & 1.0638 \\
ESCA & HNZ & $-$28,829.31 & $-$28,865.09 & Student-$t$ & 0.6807 & 1.0188 \\
\addlinespace
ISO & HNE & $-$100,096.19 
